# `DataCache` - Mise en cache des données agricoles

**Module :** `kadi.kidas.cache`  
**Classe :** `DataCache`

---

## Introduction

`DataCache` est le moteur de persistance du module kidas. Il évite de recharger et retraiter des données déjà acquises :

- **Stockage SQLite** : dans `~/.kadi/kidas_cache/` (configurable)
- **Compression zlib** : économie de 60–80% d'espace disque sur les DataFrames
- **Versioning SHA256** : détection automatique des modifications de la source
- **Historique** : chaque remplacement archive la version précédente
- **Expiration** : suppression automatique des entrées trop anciennes
- **Inspection** : taille, nombre d'entrées, clés stockées

## 1. Configuration initiale

In [1]:
import sys
import os
import pandas as pd
sys.path.append(os.path.abspath('../../'))

import os
import tempfile
import pandas as pd
import numpy as np

from kadi.kidas.cache import DataCache

# Répertoire temporaire pour les exemples
cache_dir = os.path.join(tempfile.gettempdir(), 'kidas_demo_cache')

# DataFrame agricole de démonstration
df_demo = pd.DataFrame({
    'culture': ['maïs', 'niébé', 'igname', 'sorgho', 'manioc'],
    'marche':  ['Dantokpa', 'Parakou', 'Bohicon', 'Kandi', 'Cotonou'],
    'prix_xof': [350, 500, 250, 300, 180],
    'rendement_kg': [1500.0, 800.0, 2000.0, 1200.0, 3500.0],
    'date': pd.to_datetime(['2024-01-15'] * 5),
})

print(f"Répertoire cache : {cache_dir}")

Répertoire cache : /tmp/kidas_demo_cache


## 2. Initialisation de `DataCache`

```python
DataCache(
    cache_dir: str = '~/.kadi/kidas_cache/',
    max_age_days: int = 30,
)
```

In [2]:
# Initialisation avec répertoire par défaut (~/.kadi/kidas_cache/)
cache_defaut = DataCache()
print(f"Cache par défaut : {cache_defaut.cache_dir}")
print(f"Durée max        : {cache_defaut.max_age_days} jours")

Cache par défaut : /home/dels/.kadi/kidas_cache
Durée max        : 365 jours


In [3]:
# Initialisation avec répertoire et durée personnalisés
cache = DataCache(cache_dir=cache_dir, max_age_days=365)
print(repr(cache))

## 3. `save()` - Sauvegarde d'un DataFrame dans le cache

In [4]:
cache = DataCache(cache_dir=cache_dir)

# Sauvegarde avec une clé descriptive
cle = 'prix_marches_benin_jan2024'
succes = cache.save(cle, df_demo)
print(f"Sauvegarde réussie : {succes}")

Sauvegarde réussie : True


In [5]:
# Sauvegarde avec métadonnées optionnelles
metadata_supplementaires = {
    'source': 'MAEP Bénin',
    'region': 'Sud Bénin',
    'nb_marches': 5,
}

succes = cache.save(
    'prix_avec_meta',
    df_demo,
    metadata=metadata_supplementaires,
)
print(f"Sauvegarde avec métadonnées : {succes}")

Sauvegarde avec métadonnées : True


## 4. `load()` - Chargement d'un DataFrame depuis le cache

In [6]:
# Chargement par clé
df_charge, meta_charge = cache.load('prix_marches_benin_jan2024')

if df_charge is not None:
    print(f"DataFrame chargé : {len(df_charge)} lignes × {len(df_charge.columns)} colonnes")
    print("\nMétadonnées du cache :")
    for cle_meta, val_meta in meta_charge.items():
        print(f"  {cle_meta:12s} : {val_meta}")
    df_charge

DataFrame chargé : 5 lignes × 5 colonnes

Métadonnées du cache :
  nb_rows      : 5
  nb_cols      : 5
  columns      : ['culture', 'marche', 'prix_xof', 'rendement_kg', 'date']


In [7]:
# Chargement d'une clé inexistante → None
df_absent, meta_absent = cache.load('cle_inexistante')
print(f"Clé absente → df={df_absent}, meta={meta_absent}")

Clé absente → df=None, meta=None


In [8]:
# Chargement avec métadonnées enrichies
df_meta, meta_meta = cache.load('prix_avec_meta')
print("Métadonnées personnalisées récupérées :")
if meta_meta:
    for k, v in meta_meta.items():
        print(f"  {k} : {v}")

Métadonnées personnalisées récupérées :
  source : MAEP Bénin
  region : Sud Bénin
  nb_marches : 5


## 5. `get_cached_keys()` - Lister les clés stockées

In [9]:
# Ajout de quelques entrées supplémentaires
cache.save('chirps_precip_2024', df_demo.copy())
cache.save('soilgrids_ph_benin', df_demo.copy())

# Listage de toutes les clés
cles = cache.get_cached_keys()
print(f"Entrées en cache ({len(cles)}) :")
for cle in cles:
    print(f"  - {cle}")

Entrées en cache (4) :
  - chirps_precip_2024
  - prix_avec_meta
  - prix_marches_benin_jan2024
  - soilgrids_ph_benin


## 6. `get_cache_size()` - Taille et statistiques du cache

In [10]:
taille = cache.get_cache_size()
print("=== Statistiques du cache ===")
for cle, val in taille.items():
    print(f"  {cle:15s} : {val}")

=== Statistiques du cache ===
  total_mb        : 0.027
  num_entries     : 4
  oldest_date     : 2026-07-07


## 7. `get_history()` - Historique des versions d'une entrée

In [11]:
# Simulation d'une mise à jour (2ème sauvegarde = archive la 1ère)
df_v2 = df_demo.copy()
df_v2['prix_xof'] = df_v2['prix_xof'] * 1.05  # +5% inflation

cache.save('prix_marches_benin_jan2024', df_v2)  # remplace la v1

# Consultation de l'historique
historique = cache.get_history('prix_marches_benin_jan2024')
print(f"Versions archivées : {len(historique)}")
for i, version in enumerate(historique):
    print(f"  Version {i+1} : {version.get('archived_at', 'N/A')}")

Versions archivées : 1
  Version 1 : N/A


## 8. `invalidate()` - Suppression d'une entrée du cache

In [12]:
# Suppression d'une entrée spécifique
succes_inv = cache.invalidate('chirps_precip_2024')
print(f"Entrée 'chirps_precip_2024' supprimée : {succes_inv}")

# Vérification
cles_restantes = cache.get_cached_keys()
print(f"Clés restantes : {cles_restantes}")

Entrée 'chirps_precip_2024' supprimée : True
Clés restantes : ['prix_avec_meta', 'prix_marches_benin_jan2024', 'soilgrids_ph_benin']


## 9. `invalidate_older_than()` - Purge des entrées expirées

In [13]:
# Suppression des entrées de plus de 365 jours
nb_supprimes = cache.invalidate_older_than(days=365)
print(f"Entrées supprimées (>365 jours) : {nb_supprimes}")

# Avec 0 jours → supprime tout (test de nettoyage)
nb_tout = cache.invalidate_older_than(days=0)
print(f"Entrées supprimées (>0 jours)   : {nb_tout}")

Entrées supprimées (>365 jours) : 0
Entrées supprimées (>0 jours)   : 3


## 10. `clear()` - Vider complètement le cache

In [14]:
# Rechargement de quelques entrées
cache.save('entree_1', df_demo)
cache.save('entree_2', df_demo)
print(f"Avant clear : {len(cache.get_cached_keys())} entrées")

# Vidage complet
cache.clear()
print(f"Après clear : {len(cache.get_cached_keys())} entrées")

Avant clear : 2 entrées
Après clear : 0 entrées


## 11. Pattern Cache-aside (recommandé)

Stratégie recommandée : charger depuis le cache si disponible, sinon acquérir et sauvegarder.

In [15]:
def charger_donnees_marche(marche: str, annee: int, cache: DataCache) -> pd.DataFrame:
    """Charge les données d'un marché depuis le cache ou les génère.

    Args:
        marche (str): Nom du marché cible.
        annee (int): Année des données.
        cache (DataCache): Instance du cache kidas.

    Returns:
        pd.DataFrame: Données du marché (depuis le cache ou fraîchement acquises).
    """
    # Construction de la clé de cache unique
    cle = f"marche_{marche.lower()}_{annee}"

    # Tentative de chargement depuis le cache
    df_cached, _ = cache.load(cle)
    if df_cached is not None:
        print(f"Cache HIT pour '{cle}'")
        return df_cached

    # Cache MISS : acquisition des données (simulation)
    print(f"Cache MISS pour '{cle}' → acquisition en cours...")
    df_nouveau = pd.DataFrame({
        'culture': ['maïs', 'niébé', 'igname'],
        'marche': [marche] * 3,
        'annee': [annee] * 3,
        'prix_xof': [350, 500, 250],
    })

    # Sauvegarde dans le cache
    cache.save(cle, df_nouveau)
    return df_nouveau


# Démonstration du pattern
cache = DataCache(cache_dir=cache_dir)

# 1er appel : Cache MISS (acquisition)
df_dantokpa = charger_donnees_marche('Dantokpa', 2024, cache)
print()

# 2ème appel : Cache HIT (lecture directe)
df_dantokpa_v2 = charger_donnees_marche('Dantokpa', 2024, cache)
print(f"\nDataFrame chargé : {len(df_dantokpa_v2)} lignes")

Cache MISS pour 'marche_dantokpa_2024' → acquisition en cours...

Cache HIT pour 'marche_dantokpa_2024'

DataFrame chargé : 3 lignes


## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `save(key, df, metadata)` | Sauvegarder un DataFrame (pickle + zlib) |
| `load(key)` | Charger un DataFrame et ses métadonnées |
| `get_cached_keys()` | Lister toutes les clés stockées |
| `get_cache_size()` | Statistiques d'occupation du cache |
| `get_history(key)` | Historique des versions pour une clé |
| `invalidate(key)` | Supprimer une entrée spécifique |
| `invalidate_older_than(days)` | Purger les entrées expirées |
| `clear()` | Vider complètement le cache |